In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

workspace_root = Path.cwd().parent
env_paths = (
    Path.cwd() / ".env",
    workspace_root / ".env",
    workspace_root / "Langchain_Basics" / ".env",
)

for env_path in env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f"Loaded environment from: {env_path}")
        break
else:
    print("No .env file found. Create Agents/.env or workspace-root/.env.")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "LangChainTrainings-Agents")

if os.getenv("LANGSMITH_API_KEY"):
    print(f"LangSmith tracing enabled for project: {os.environ['LANGSMITH_PROJECT']}")
else:
    print("Add LANGSMITH_API_KEY to .env to enable LangSmith tracing.")

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2500,
)

In [ ]:
# Wikipedia tool with retry handling

import json
import time

from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain.tools import tool

wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=2,
        doc_content_chars_max=4000,
    )
)


def wikipedia_invoke_with_retry(query, max_attempts=3):
    for attempt in range(max_attempts):
        try:
            return wikipedia.invoke(query)
        except Exception as error:
            if attempt == max_attempts - 1:
                return (
                    "Wikipedia is temporarily unavailable right now, "
                    f"so I cannot fetch live results. Error: {error}"
                )
            time.sleep(2 ** attempt)


@tool
def wikipedia_tool(query: str) -> str:
    """Search Wikipedia for a query and return a summary if the live Wikipedia API is available."""
    return wikipedia_invoke_with_retry(query)


# Creatin custome tools


@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def substarct(a: int, b: int) -> int:
    """Add two numbers."""
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """Add two numbers."""
    return a * b

print(add.invoke({"a":10, "b": 20}))  # Example usage of the custom tool

tools = [wikipedia_tool, add, substarct, multiply]


In [ ]:
# from langchain.agents import create_agent
# from langchain_core.messages import HumanMessage

# # Create the agent
# agent = create_agent(
#     model=llm,
#     tools=tools,
#     system_prompt="You are a helpful assistant that can answer questions and perform calculations.",
# )

# # User query
# query = (
#     "What is the capital of India? "
#     "Also, add 10 and 20, then multiply the result by 2."
# )

# # Invoke the agent
# result = agent.invoke(
#     {
#         "messages": [
#             HumanMessage(content=query)
#         ]
#     }
# )

# # for result in result["messages"]:
# #     print(result.content)


# # Print final response
# print(result["messages"][-1].content)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

promtp_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant that can answer questions and perform calculations."),
    ("user", "whats the sum of 10 and 20, then multiply the result by 2."),
    ("user", "whats the largest city in India?"),
    ("user", "whats silicon city of India?"),
    ("user", "whats Garden city of India?"),
    ("user", "Which is the cleanest city of India?"),
    ("user","give me both answers in json format")
])


result = agent.invoke({"messages": promtp_template.format_messages()})
    
print(result["messages"][-1].content)
    

In [ ]:
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import (
    create_async_playwright_browser
)
import nest_asyncio

nest_asyncio.apply()

In [ ]:
async_browser = create_async_playwright_browser()
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
browser_tools = toolkit.get_tools()

tools = [wikipedia_tool, add, substarct, multiply, *browser_tools]
tools

In [ ]:
from langchain_core.tools import tool
from playwright.sync_api import sync_playwright


@tool
def extract_webpage(url: str) -> str:
    """Open a webpage and return its visible text."""

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()

        page.goto(url, wait_until="domcontentloaded")

        text = page.locator("body").inner_text()

        browser.close()

        return text

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[extract_webpage]
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Open https://www.google.com/ and tell me the main text"
        }
    ]
})

print(result["messages"][-1].content)

In [15]:
import asyncio

loop = asyncio.get_running_loop()

print(type(loop))
print(type(loop).__module__)

<class 'asyncio.windows_events._WindowsSelectorEventLoop'>
asyncio.windows_events


In [ ]:
import asyncio
import sys

if sys.platform == "win32":
    asyncio.set_event_loop_policy(
        asyncio.WindowsProactorEventLoopPolicy()
    )

In [ ]:
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()

    await page.goto("https://example.com")
    print(await page.title())

    await browser.close()